# Dropout, Batch Normalization, and Ensembling in CNNs (PyTorch)

In [1]:
# --- 1. Setup ---
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torchvision import datasets, transforms
from torch.utils.data import DataLoader, random_split
import matplotlib.pyplot as plt

In [2]:
# Device config
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)


Using device: cuda


In [ ]:
# Set manual seed for reproducibility
torch.manual_seed(42)

In [10]:
# Create a dummy input tensor
x = torch.ones(10)

# Define a dropout layer with 50% dropout rate
dropout = nn.Dropout(p=0.3)

print("Original Input:")
print(x)

# Apply dropout in training mode
dropout.train()  # This is the default, but showing explicitly
out_train = dropout(x)

print("\nAfter Dropout (Training Mode):")
print(out_train)

# Apply dropout in evaluation mode (no dropout applied)
dropout.eval()
out_eval = dropout(x)

print("\nAfter Dropout (Evaluation Mode):")
print(out_eval)

Original Input:
tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])

After Dropout (Training Mode):
tensor([0.0000, 1.4286, 1.4286, 1.4286, 1.4286, 1.4286, 1.4286, 0.0000, 1.4286,
        1.4286])

After Dropout (Evaluation Mode):
tensor([1., 1., 1., 1., 1., 1., 1., 1., 1., 1.])


In [14]:
# --- 2. Load MNIST Dataset ---
transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.1307,), (0.3081,))
])

train_dataset = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_dataset = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_dataset, batch_size=60_000, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=1000, shuffle=False)


In [15]:
for x,y in train_loader:
  print(x.shape)

torch.Size([60000, 1, 28, 28])


In [17]:
torch.std(x)

tensor(0.3081)

In [12]:
64*12*12

9216

In [4]:
# --- 3. Define CNN Models ---
# Base CNN (no dropout / no batchnorm)
class BaseCNN(nn.Module):
    def __init__(self):
        super(BaseCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.fc1 = nn.Linear(9216, 128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

# CNN with Dropout
class DropoutCNN(nn.Module):
    def __init__(self):
        super(DropoutCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.fc1 = nn.Linear(9216, 128)
        self.dropout = nn.Dropout(0.5)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.conv1(x))
        x = F.relu(self.conv2(x))
        x = F.max_pool2d(x, 2)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.dropout(x)
        x = self.fc2(x)
        return x

# CNN with Batch Normalization
class BatchNormCNN(nn.Module):
    def __init__(self):
        super(BatchNormCNN, self).__init__()
        self.conv1 = nn.Conv2d(1, 32, 3, 1)
        self.bn1 = nn.BatchNorm2d(32)
        self.conv2 = nn.Conv2d(32, 64, 3, 1)
        self.bn2 = nn.BatchNorm2d(64)
        self.fc1 = nn.Linear(9216, 128)
        # self.bn3 = nn.BatchNorm1d(128)
        self.fc2 = nn.Linear(128, 10)

    def forward(self, x):
        x = F.relu(self.bn1(self.conv1(x)))
        x = F.relu(self.bn2(self.conv2(x)))
        x = F.max_pool2d(x, 2)
        x = torch.flatten(x, 1)
        x = F.relu(self.fc1(x))
        x = self.fc2(x)
        return x

In [5]:
# --- 4. Training and Evaluation Functions ---
def train(model, train_loader, optimizer, epoch):
    model.train()
    for batch_idx, (data, target) in enumerate(train_loader):
        data, target = data.to(device), target.to(device)
        optimizer.zero_grad()
        output = model(data)
        loss = F.cross_entropy(output, target)
        loss.backward()
        optimizer.step()
        if batch_idx % 100 == 0:
            print(f"Train Epoch: {epoch} [{batch_idx*len(data)}/{len(train_loader.dataset)}]\tLoss: {loss.item():.6f}")

def test(model, test_loader):
    model.eval()
    test_loss = 0
    correct = 0
    with torch.no_grad():
        for data, target in test_loader:
            data, target = data.to(device), target.to(device)
            output = model(data)
            test_loss += F.cross_entropy(output, target, reduction='sum').item()
            pred = output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
    test_loss /= len(test_loader.dataset)
    accuracy = 100. * correct / len(test_loader.dataset)
    print(f"Test set: Average loss: {test_loss:.4f}, Accuracy: {correct}/{len(test_loader.dataset)} ({accuracy:.2f}%)")
    return accuracy


In [6]:
# --- 5. Train & Compare Models ---
models = {
    "BaseCNN": BaseCNN().to(device),
    "DropoutCNN": DropoutCNN().to(device),
    "BatchNormCNN": BatchNormCNN().to(device)
}

results = {}
for name, model in models.items():
    print("\nTraining", name)
    optimizer = optim.Adam(model.parameters(), lr=0.001)
    for epoch in range(1, 4):  # fewer epochs for demo
        train(model, train_loader, optimizer, epoch)
        acc = test(model, test_loader)
    results[name] = acc

print("\nFinal Results:")
for k, v in results.items():
    print(f"{k}: {v:.2f}%")



Training BaseCNN
Train Epoch: 1 [0/60000]	Loss: 2.302767
Train Epoch: 1 [6400/60000]	Loss: 0.121761
Train Epoch: 1 [12800/60000]	Loss: 0.104134
Train Epoch: 1 [19200/60000]	Loss: 0.068240
Train Epoch: 1 [25600/60000]	Loss: 0.210681
Train Epoch: 1 [32000/60000]	Loss: 0.225158
Train Epoch: 1 [38400/60000]	Loss: 0.030050
Train Epoch: 1 [44800/60000]	Loss: 0.081194
Train Epoch: 1 [51200/60000]	Loss: 0.238200
Train Epoch: 1 [57600/60000]	Loss: 0.056496
Test set: Average loss: 0.0501, Accuracy: 9845/10000 (98.45%)
Train Epoch: 2 [0/60000]	Loss: 0.015967
Train Epoch: 2 [6400/60000]	Loss: 0.043067
Train Epoch: 2 [12800/60000]	Loss: 0.035320
Train Epoch: 2 [19200/60000]	Loss: 0.059997
Train Epoch: 2 [25600/60000]	Loss: 0.046637
Train Epoch: 2 [32000/60000]	Loss: 0.024464
Train Epoch: 2 [38400/60000]	Loss: 0.010442
Train Epoch: 2 [44800/60000]	Loss: 0.005926
Train Epoch: 2 [51200/60000]	Loss: 0.001988
Train Epoch: 2 [57600/60000]	Loss: 0.018737
Test set: Average loss: 0.0310, Accuracy: 9903/100

In [7]:
# --- 6. Ensembling ---
print("\nEvaluating Ensemble of all 3 models")
def ensemble_predict(models, loader):
    for model in models:
        model.eval()
    correct = 0
    with torch.no_grad():
        for data, target in loader:
            data, target = data.to(device), target.to(device)
            outputs = [F.softmax(m(data), dim=1) for m in models]
            avg_output = torch.mean(torch.stack(outputs), dim=0)
            pred = avg_output.argmax(dim=1, keepdim=True)
            correct += pred.eq(target.view_as(pred)).sum().item()
    accuracy = 100. * correct / len(loader.dataset)
    return accuracy

ensemble_acc = ensemble_predict(list(models.values()), test_loader)
print(f"Ensemble Accuracy: {ensemble_acc:.2f}%")



Evaluating Ensemble of all 3 models
Ensemble Accuracy: 99.21%
